# Don't Let Your Next iPhone Catch Fire!
## Discovering Thermally and Mechanically Stable Li-Ion Battery Separator Polymers

Every lithium-ion battery contains a thin polymer separator that prevents the electrodes from short-circuiting. If this film melts or shrinks during overheating, the battery can catch fire. Most commercial separators use PE or PP, which melt at just 130-165°C — a known weak point during thermal runaway.

In this project, we train a KNN classifier on 39 known separator polymers to predict safety from material properties, then screen 101 untested polymers to discover new candidates.

---
## Part 1: Building the Training Dataset

We combine 3 data sources:
1. **PMC literature** (web scraped) — identified separator polymers + shutdown temperatures → safety labels
2. **Celgard** (web scraped HTML + PDFs) — confirmed commercial separator polymers
3. **OpenPoly database** (downloaded) — 6 material properties for 741 polymers

All merged by polymer name → 39 labeled training polymers (25 safe, 14 unsafe)

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load the OpenPoly polymer database
df_polymers = pd.read_csv("data/final_polymer_properties_fromliterature.csv")
print(f"OpenPoly database: {len(df_polymers)} polymers")
df_polymers.head()

### Step 1: Define known separator polymers with safety labels

Safety labels come from published thermal stability data (PMC review articles):
- **safe** = maintains structural integrity above 150°C
- **unsafe** = fails at or below 150°C

The 150°C threshold is based on industry abuse test standards (UL 2054, IEC 62133).

In [ ]:
separator_polymers = pd.DataFrame([
    # --- Polyolefins ---
    {"polymer": "PP", "shutdown_temp_C": 170, "safety": "safe"},
    {"polymer": "PE", "shutdown_temp_C": 135, "safety": "unsafe"},
    {"polymer": "UHMWPE", "shutdown_temp_C": 135, "safety": "unsafe"},
    {"polymer": "HDPE", "shutdown_temp_C": 130, "safety": "unsafe"},
    {"polymer": "LDPE", "shutdown_temp_C": 110, "safety": "unsafe"},

    # --- Fluoropolymers ---
    {"polymer": "PVDF", "shutdown_temp_C": 160, "safety": "safe"},
    {"polymer": "PTFE", "shutdown_temp_C": 327, "safety": "safe"},

    # --- High-performance polymers ---
    {"polymer": "PAN", "shutdown_temp_C": 300, "safety": "safe"},
    {"polymer": "PI", "shutdown_temp_C": 350, "safety": "safe"},
    {"polymer": "PPS", "shutdown_temp_C": 300, "safety": "safe"},
    {"polymer": "PEI", "shutdown_temp_C": 280, "safety": "safe"},
    {"polymer": "PES", "shutdown_temp_C": 220, "safety": "safe"},
    {"polymer": "PSF", "shutdown_temp_C": 185, "safety": "safe"},

    # --- Engineering polymers ---
    {"polymer": "PET", "shutdown_temp_C": 250, "safety": "safe"},
    {"polymer": "PA6", "shutdown_temp_C": 220, "safety": "safe"},
    {"polymer": "PC", "shutdown_temp_C": 230, "safety": "safe"},
    {"polymer": "POM", "shutdown_temp_C": 175, "safety": "safe"},
    {"polymer": "nylon", "shutdown_temp_C": 220, "safety": "safe"},

    # --- Hydrophilic / gel polymers ---
    {"polymer": "PVA", "shutdown_temp_C": 230, "safety": "safe"},
    {"polymer": "PMMA", "shutdown_temp_C": 105, "safety": "unsafe"},
    {"polymer": "PAA", "shutdown_temp_C": 200, "safety": "safe"},
    {"polymer": "PVP", "shutdown_temp_C": 60, "safety": "unsafe"},
    {"polymer": "PEG", "shutdown_temp_C": 60, "safety": "unsafe"},
    {"polymer": "PEO", "shutdown_temp_C": 65, "safety": "unsafe"},

    # --- Biodegradable / low-stability ---
    {"polymer": "PLA", "shutdown_temp_C": 160, "safety": "safe"},
    {"polymer": "PBS", "shutdown_temp_C": 115, "safety": "unsafe"},
    {"polymer": "PCL", "shutdown_temp_C": 60, "safety": "unsafe"},

    # --- Bio-based ---
    {"polymer": "cellulose", "shutdown_temp_C": 260, "safety": "safe"},
    {"polymer": "chitosan", "shutdown_temp_C": 200, "safety": "safe"},

    # --- Others ---
    {"polymer": "PS", "shutdown_temp_C": 100, "safety": "unsafe"},
    {"polymer": "PVC", "shutdown_temp_C": 100, "safety": "unsafe"},
    {"polymer": "PMP", "shutdown_temp_C": 235, "safety": "safe"},

    # --- Duplicate names in OpenPoly ---
    {"polymer": "polypropylene", "shutdown_temp_C": 170, "safety": "safe"},
    {"polymer": "polyethylene", "shutdown_temp_C": 135, "safety": "unsafe"},
    {"polymer": "polyimide", "shutdown_temp_C": 350, "safety": "safe"},
    {"polymer": "polycarbonate", "shutdown_temp_C": 230, "safety": "safe"},
    {"polymer": "polystyrene", "shutdown_temp_C": 100, "safety": "unsafe"},
    {"polymer": "poly(vinylidene fluoride)", "shutdown_temp_C": 160, "safety": "safe"},
    {"polymer": "polyacrylonitrile", "shutdown_temp_C": 300, "safety": "safe"},
    {"polymer": "polyethersulfone", "shutdown_temp_C": 220, "safety": "safe"},
    {"polymer": "polysulfone", "shutdown_temp_C": 185, "safety": "safe"},
    {"polymer": "cellulose acetate", "shutdown_temp_C": 230, "safety": "safe"},
])

print(f"Compiled {len(separator_polymers)} separator polymers")
separator_polymers["safety"].value_counts()

### Step 2: Join with OpenPoly for material properties

We use 6 material properties as model features:
- **Thermal**: Tg (glass transition temp), Tm (melting temp), Td (decomposition temp)
- **Mechanical**: Tensile Strength, Young's Modulus, Elongation at Break

In [ ]:
key_features = ["Tg (K)", "Tm (K)", "Td (K)",
                "Tensile Strength (MPa)", "Young's Modulus (MPa)",
                "Elongation at Break (%)"]

df_polymer_props = df_polymers[["Name"] + key_features].copy()

# Deduplicate: keep row with most non-null features per polymer
df_polymer_props["n_features"] = df_polymer_props[key_features].notna().sum(axis=1)
df_polymer_props = df_polymer_props.sort_values("n_features", ascending=False)
df_polymer_props = df_polymer_props.drop_duplicates(subset="Name", keep="first")
df_polymer_props = df_polymer_props.drop(columns="n_features")

# Merge safety labels with material properties
df_training = separator_polymers.merge(
    df_polymer_props,
    left_on="polymer",
    right_on="Name",
    how="left"
)

# Drop rows without complete material properties
df_training = df_training.dropna(subset=key_features)
df_training = df_training.drop(columns="Name")

print(f"Training set: {len(df_training)} polymers with complete data")
print(f"\nSafety distribution:")
print(df_training["safety"].value_counts())

In [ ]:
# View the training data
df_training[["polymer", "shutdown_temp_C", "safety"] + key_features]

In [ ]:
# Save training data
df_training.to_csv("data/training_data.csv", index=False)
print("Saved to data/training_data.csv")

---
## Part 2: Training the KNN Model

We build a pipeline with StandardScaler + KNeighborsClassifier, evaluate with cross-validation, and tune hyperparameters with GridSearchCV.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import confusion_matrix, f1_score

In [ ]:
# Set up features and labels
X_train = df_training[key_features]
y_train = df_training["safety"]

print(f"Training set: {len(X_train)} polymers, {len(key_features)} features")
print(f"Labels: {y_train.value_counts().to_dict()}")

### Step 1: Build pipeline and cross-validate

In [ ]:
# Pipeline: scale features, then classify with KNN
pipeline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5, metric="euclidean")
)

# Cross-validation: accuracy
scores_acc = cross_val_score(pipeline, X_train, y_train, scoring="accuracy", cv=5)
print(f"Accuracy:  {scores_acc.mean():.3f} (+/- {scores_acc.std():.3f})")

# Cross-validation: F1 macro (better for imbalanced classes)
scores_f1 = cross_val_score(pipeline, X_train, y_train, scoring="f1_macro", cv=5)
print(f"F1 (macro): {scores_f1.mean():.3f} (+/- {scores_f1.std():.3f})")

### Step 2: GridSearchCV to find best k and distance metric

In [ ]:
grid_cv = GridSearchCV(
    pipeline,
    param_grid={
        "kneighborsclassifier__n_neighbors": range(1, 12),
        "kneighborsclassifier__metric": ["euclidean", "manhattan"],
    },
    scoring="f1_macro",
    cv=5
)

grid_cv.fit(X_train, y_train)
print(f"Best params: {grid_cv.best_params_}")
print(f"Best F1 (macro): {grid_cv.best_score_:.3f}")

### Model Selection Plot

In [ ]:
import matplotlib.pyplot as plt

results = pd.DataFrame(grid_cv.cv_results_)
fig, ax = plt.subplots()
for metric in ["euclidean", "manhattan"]:
    mask = results["param_kneighborsclassifier__metric"] == metric
    subset = results[mask].sort_values("param_kneighborsclassifier__n_neighbors")
    subset.plot.line(
        x="param_kneighborsclassifier__n_neighbors",
        y="mean_test_score",
        label=metric,
        ax=ax
    )

# Label the best F1 point
best_k = grid_cv.best_params_["kneighborsclassifier__n_neighbors"]
best_metric = grid_cv.best_params_["kneighborsclassifier__metric"]
best_f1 = grid_cv.best_score_
ax.annotate(f"Best: k={best_k}, {best_metric}\nF1 = {best_f1:.3f}",
            xy=(best_k, best_f1),
            xytext=(best_k - 2, best_f1 - 0.06),
            fontsize=10, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="black"),
            bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.8))
ax.scatter([best_k], [best_f1], color="black", zorder=5, s=60)

ax.set_xticks(range(1, 12))
ax.set_xlabel("k (number of neighbors)")
ax.set_ylabel("F1 Score (macro)")
ax.set_title("Model Selection: F1 vs k")
ax.legend()
plt.show()

### Step 3: Training confusion matrix

In [ ]:
best_pipeline = grid_cv.best_estimator_
y_train_pred = best_pipeline.predict(X_train)

cm = confusion_matrix(y_train, y_train_pred, labels=["safe", "unsafe"])
print("Confusion Matrix (rows=actual, cols=predicted):")
print(f"Labels: [safe, unsafe]")
print(cm)

# Per-class metrics
print("\nPer-Class Metrics:")
for label in ["safe", "unsafe"]:
    y_true_binary = (y_train == label)
    y_pred_binary = (y_train_pred == label)
    tp = (y_true_binary & y_pred_binary).sum()
    fp = (~y_true_binary & y_pred_binary).sum()
    fn = (y_true_binary & ~y_pred_binary).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 / (1/precision + 1/recall) if (precision > 0 and recall > 0) else 0
    print(f"  {label:10s}  precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}")

---
## Part 3: Screening Untested Polymers

We use the trained model to predict safety for 101 polymers from OpenPoly that are NOT in our training set.

In [ ]:
# Get polymers with complete features that are NOT in training set
training_names = df_training["polymer"].str.lower().tolist()
df_screen = df_polymers.dropna(subset=key_features).copy()
df_screen = df_screen[~df_screen["Name"].str.lower().isin(training_names)]

print(f"Polymers available to screen: {len(df_screen)}")

# Predict safety
X_screen = df_screen[key_features]
df_screen["predicted_safety"] = best_pipeline.predict(X_screen)
df_screen["predicted_proba_safe"] = best_pipeline.predict_proba(X_screen)[:,
    list(best_pipeline.classes_).index("safe")]

df_screen = df_screen.sort_values("predicted_proba_safe", ascending=False)

print(f"\nScreening Summary:")
print(df_screen["predicted_safety"].value_counts())

In [ ]:
# Top 20 predicted safe candidates
top_safe = df_screen[df_screen["predicted_safety"] == "safe"].head(20)
top_safe[["Name", "predicted_safety", "predicted_proba_safe"] + key_features]

In [ ]:
# Save screening results
df_screen.to_csv("data/screening_results.csv", index=False)
print("Saved to data/screening_results.csv")

### Screening Results Plot

In [ ]:
import plotly.express as px

# Build combined DataFrame for plotly
df_screen_plot = df_screen[["Tg (K)", "Tm (K)", "Name", "predicted_safety"]].copy()
df_screen_plot["group"] = "Predicted " + df_screen_plot["predicted_safety"]

df_train_plot = df_training[["Tg (K)", "Tm (K)", "polymer", "safety"]].copy()
df_train_plot = df_train_plot.rename(columns={"polymer": "Name"})
df_train_plot["group"] = "Known " + df_train_plot["safety"]

df_all_plot = pd.concat([df_screen_plot, df_train_plot], ignore_index=True)

fig = px.scatter(df_all_plot, x="Tg (K)", y="Tm (K)", color="group",
                 hover_data=["Name"],
                 title="Separator Safety Prediction: Known vs. Screened Polymers",
                 color_discrete_map={
                     "Predicted safe": "lightgreen",
                     "Predicted unsafe": "lightsalmon",
                     "Known safe": "green",
                     "Known unsafe": "red",
                 })
fig.update_layout(width=700, height=450)
fig.update_xaxes(title_text="Glass Transition Temperature (K)", range=[150, 600], dtick=50)
fig.update_yaxes(title_text="Melting Temperature (K)", range=[280, 650])
fig.show()

---
## Part 4: K-Means Clustering

We use unsupervised K-means clustering to see if polymers naturally group by their material properties, and whether those groups align with safety labels.

In [ ]:
from sklearn.cluster import KMeans

# Use all polymers with complete data
cluster_features = key_features
df_all = df_polymers.dropna(subset=cluster_features).copy()
X_all = df_all[cluster_features]

print(f"Clustering {len(X_all)} polymers with complete data")

### Elbow Plot — choosing k

In [ ]:
# Scale the data
scaler = StandardScaler()
X_all_scaled = scaler.fit_transform(X_all)

# Try different numbers of clusters
ks = range(2, 11)
inertias = []
for k in ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_all_scaled)
    inertias.append(model.inertia_)

# Elbow plot
df_elbow = pd.DataFrame({"k": list(ks), "Inertia": inertias})
ax = df_elbow.plot.line(x="k", y="Inertia", marker="o", legend=False)
ax.set_xlabel("k (number of clusters)")
ax.set_ylabel("Inertia (within-cluster sum of squares)")
ax.set_title("Elbow Plot for K-Means Clustering")
plt.show()

### Fit K-Means with k=3 and compare to safety labels

In [ ]:
# Fit K-Means with k=3 using a pipeline
pipeline_km = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=3, random_state=42, n_init=10)
)
pipeline_km.fit(X_all)

kmeans_model = pipeline_km.named_steps["kmeans"]
df_all["cluster"] = kmeans_model.labels_
df_all["Cluster"] = df_all["cluster"].astype(str)

print("Cluster sizes:")
print(pd.Series(kmeans_model.labels_).value_counts().sort_index())

In [ ]:
# Compare clusters to known safety labels
df_training_clustered = df_training.merge(
    df_all[["Name", "cluster"]],
    left_on="polymer",
    right_on="Name",
    how="left"
)

print("Cluster vs. Safety Label (known separator polymers):")
pd.crosstab(df_training_clustered["cluster"], df_training_clustered["safety"])

### Clusters vs. Safety Labels Plot

In [ ]:
cluster_colors = {"0": "#636EFA", "1": "#EF553B", "2": "#00CC96"}

fig = px.scatter(df_all, x="Tg (K)", y="Tm (K)", color="Cluster",
                 hover_data=["Name"],
                 color_discrete_map=cluster_colors,
                 title="K-Means Clusters vs. Known Safety Labels<br>(Clustered on all 6 properties, visualized in Tg vs Tm space)")
fig.update_traces(marker=dict(size=6, opacity=0.8))
fig.update_xaxes(title_text="Glass Transition Temperature (K)")
fig.update_yaxes(title_text="Melting Temperature (K)")

# Overlay known polymers with different shapes
shape_map = {"safe": "square", "unsafe": "x"}
for label in ["safe", "unsafe"]:
    subset = df_training_clustered[df_training_clustered["safety"] == label]
    subset = subset.dropna(subset=["cluster"])
    colors = [cluster_colors[str(int(c))] for c in subset["cluster"]]
    fig.add_scatter(
        x=subset["Tg (K)"], y=subset["Tm (K)"],
        mode="markers",
        name=f"Known {label}",
        marker=dict(size=8, color=colors,
                    symbol=shape_map[label],
                    line=dict(width=1.5, color="black")),
        text=subset["polymer"], hoverinfo="text")

fig.update_xaxes(range=[df_all["Tg (K)"].min() - 20, 600])
fig.update_yaxes(range=[df_all["Tm (K)"].min() - 20, 650])
fig.show()

---
## Part 5: Cross-Validation Confusion Matrix

The CV confusion matrix is more honest than training — each polymer is predicted by a model that never saw it during training.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score)

# Re-run GridSearchCV to get best model
pipeline_knn = make_pipeline(StandardScaler(), KNeighborsClassifier())
grid_cv = GridSearchCV(
    pipeline_knn,
    param_grid={
        "kneighborsclassifier__n_neighbors": range(1, 12),
        "kneighborsclassifier__metric": ["euclidean", "manhattan"],
    },
    scoring="f1_macro",
    cv=5
)
grid_cv.fit(X_train, y_train)
best_pipeline = grid_cv.best_estimator_

# Get out-of-fold predictions
y_cv_pred = cross_val_predict(best_pipeline, X_train, y_train, cv=5)

labels = ["safe", "unsafe"]
cm_cv = confusion_matrix(y_train, y_cv_pred, labels=labels)
print("CV Confusion Matrix (rows=actual, cols=predicted):")
print(cm_cv)

In [ ]:
# Visualize: confusion matrix + metrics table side by side
fig, (ax_cm, ax_metrics) = plt.subplots(1, 2, figsize=(12, 5),
                                        gridspec_kw={"width_ratios": [1, 0.8]})

# Left: confusion matrix heatmap
disp_cv = ConfusionMatrixDisplay(cm_cv, display_labels=labels)
disp_cv.plot(ax=ax_cm, cmap="Blues", values_format="d")
ax_cm.set_title("Cross-Validation Confusion Matrix", fontsize=14)
ax_cm.set_xlabel("Predicted Label", fontsize=13)
ax_cm.set_ylabel("True Label", fontsize=13)

# Right: precision, recall, F1 table
ax_metrics.axis("off")
cv_metrics_data = []
for label in labels:
    p = precision_score(y_train, y_cv_pred, pos_label=label)
    r = recall_score(y_train, y_cv_pred, pos_label=label)
    f = f1_score(y_train, y_cv_pred, pos_label=label)
    cv_metrics_data.append([label, f"{p:.3f}", f"{r:.3f}", f"{f:.3f}"])

table = ax_metrics.table(
    cellText=cv_metrics_data,
    colLabels=["Class", "Precision", "Recall", "F1 Score"],
    loc="center", cellLoc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(15)
table.scale(1, 2.2)

cv_macro_f1 = f1_score(y_train, y_cv_pred, average="macro")
ax_metrics.set_title("Per-Class Metrics (CV)", fontsize=14)
ax_metrics.text(0.5, 0.15, f"Cross-Val Macro F1 = {cv_macro_f1:.3f}",
                transform=ax_metrics.transAxes,
                ha="center", fontsize=16, fontweight="bold")

fig.suptitle("KNN Separator Safety Classifier \u2014 Cross-Validation Performance",
             fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# Key metrics
safe_precision = precision_score(y_train, y_cv_pred, pos_label="safe")
unsafe_recall = recall_score(y_train, y_cv_pred, pos_label="unsafe")
print(f"Safe precision (CV): {safe_precision:.3f}")
print(f"  -> When the model predicts 'safe', it's correct {safe_precision*100:.0f}% of the time")
print(f"\nUnsafe recall (CV): {unsafe_recall:.3f}")
print(f"  -> The model catches {unsafe_recall*100:.0f}% of unsafe polymers")
print(f"\nAs a screening tool, precision matters most — missed unsafe polymers")
print(f"would be caught during lab testing before going into a battery.")

---
## Part 6: Top Candidates & Radar Chart

We visualize the top predicted candidates and compare their material properties against known high-performance separators (PI and PAN).

In [ ]:
# Load screening results
df_screen = pd.read_csv("data/screening_results.csv")

# Get candidates predicted safe (>50% probability)
df_top = df_screen[df_screen["predicted_proba_safe"] > 0.5].copy()
df_top = df_top.sort_values("predicted_proba_safe", ascending=False)

# Split by confidence level
top_high = df_top[df_top["predicted_proba_safe"] == 1.0].copy()
top_mod = df_top[df_top["predicted_proba_safe"] < 1.0].copy()

print(f"100% confidence candidates: {len(top_high)}")
print(f"50-99% confidence candidates: {len(top_mod)}")
top_high[["Name", "predicted_proba_safe"] + key_features]

### Top Candidates Plot

In [ ]:
top_high_plot = top_high[["Tg (K)", "Tm (K)", "Name"]].copy()
top_high_plot["group"] = "Top candidates (100% confidence)"

top_mod_plot = top_mod[["Tg (K)", "Tm (K)", "Name"]].copy()
top_mod_plot["group"] = "Candidates (50-99% confidence)"

df_plot = pd.concat([top_high_plot, top_mod_plot], ignore_index=True)

fig = px.scatter(df_plot, x="Tg (K)", y="Tm (K)", color="group",
                 hover_data=["Name"],
                 title="Top Predicted Safe Separator Polymer Candidates",
                 color_discrete_map={
                     "Top candidates (100% confidence)": "dodgerblue",
                     "Candidates (50-99% confidence)": "lightskyblue",
                 })

# Label top candidates
label_offsets = {
    "PEKK": (30, -25), "PEN": (25, 15), "PGA": (-60, -25),
    "polyimides": (30, 10), "PA 6": (-55, 20), "PHB": (-50, -20),
    "P(3HB)": (30, -15), "PLLA": (-55, 15), "PBO": (25, 20),
    "nylon 6": (30, -10),
}

for _, row in top_high.iterrows():
    if row["Name"] in label_offsets:
        ax_off, ay_off = label_offsets[row["Name"]]
        fig.add_annotation(
            x=row["Tg (K)"], y=row["Tm (K)"],
            text=row["Name"], showarrow=True,
            arrowhead=0, ax=ax_off, ay=ay_off,
            font=dict(size=10),
            bgcolor="white", bordercolor="gray", borderpad=2,
        )

fig.update_layout(width=700, height=450)
fig.update_traces(marker=dict(size=9))
fig.update_xaxes(title_text="Glass Transition Temperature (K)", range=[150, 600], dtick=50)
fig.update_yaxes(title_text="Melting Temperature (K)", range=[280, 650])
fig.show()

### Radar Chart — Candidates vs. Known Separators (PI and PAN)

We normalize all 6 properties to 0-1 (same idea as StandardScaler) so they're comparable on the same axes, then use `px.line_polar()` to create a radar chart.

In [ ]:
radar_features = ["Tg (K)", "Tm (K)", "Td (K)",
                  "Tensile Strength (MPa)", "Young's Modulus (MPa)",
                  "Elongation at Break (%)"]
radar_labels = ["Glass Transition Temp", "Melting Temp",
                "Decomposition Temp", "Tensile Strength",
                "Young's Modulus", "Elongation at Break"]

# Prepare top candidate data
top_100 = top_high[top_high["predicted_proba_safe"] == 1.0]
top_for_bar = top_100.drop_duplicates(subset="Name").set_index("Name")

candidate_names = ["polyimides", "PEKK", "PGA", "PEN", "PA 6"]

# Build DataFrame — baselines first so they render behind candidates
radar_rows = []

# Add PI and PAN as known baselines
pi = df_training[df_training["polymer"] == "polyimide"].iloc[0]
radar_rows.append({"polymer": "PI (Known)", **{f: pi[f] for f in radar_features}})

pan = df_training[df_training["polymer"] == "PAN"].iloc[0]
radar_rows.append({"polymer": "PAN (Known)", **{f: pan[f] for f in radar_features}})

for name in candidate_names:
    if name in top_for_bar.index:
        row = top_for_bar.loc[name, radar_features]
        radar_rows.append({"polymer": name, **row.to_dict()})

df_radar = pd.DataFrame(radar_rows)

# Normalize each feature to 0-1
for f in radar_features:
    fmin = df_radar[f].min()
    fmax = df_radar[f].max()
    df_radar[f] = (df_radar[f] - fmin) / (fmax - fmin) if fmax > fmin else 0.5

# Reshape to long format for plotly
df_radar_long = df_radar.melt(id_vars="polymer", value_vars=radar_features,
                              var_name="property", value_name="value")

# Replace column names with readable labels
label_map = dict(zip(radar_features, radar_labels))
df_radar_long["property"] = df_radar_long["property"].map(label_map)

fig = px.line_polar(df_radar_long, r="value", theta="property",
                    color="polymer", line_close=True,
                    color_discrete_map={
                        "PI (Known)": "#404040",
                        "PAN (Known)": "#a0a0a0",
                    },
                    title="Top Candidates vs. Current Separators (Normalized)")
fig.update_traces(fill="toself", opacity=0.6, marker=dict(size=8))
fig.update_layout(width=900, height=650,
                  polar=dict(radialaxis=dict(visible=False),
                             angularaxis=dict(tickfont=dict(size=14))),
                  legend=dict(y=1.15, x=1.15, xanchor="left", font=dict(size=13)),
                  title_font_size=16)
fig.show()

### Radar Chart Analysis

- **PI (Known)** excels thermally (highest Tg, high Td) — safe primarily due to extreme heat resistance
- **PAN (Known)** excels mechanically (highest Young's Modulus) — safe primarily due to rigidity that resists dendrite puncture
- **PEKK and PEN** are the most well-rounded candidates — strong across both thermal and mechanical properties
- **PGA** is mechanically extreme (very high tensile strength and modulus) but thermally weaker
- **PA 6** has high elongation but lower Tg — more flexible, different trade-off

Candidates closest to PI on the radar (polyimides, PEKK, PEN) are the strongest bets for lab testing.